In [ ]:
import esda
import libpysal
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

from analysis.src.data import complete_case_mask
from analysis.src.io import experiment_paths, load_data
from analysis.src.constants import ADJUSTMENT_SETS, OUTCOMES, RANDOM_STATE

In [ ]:
OUT = experiment_paths('spatial/borough_fixed_effects')
DISEASES = OUTCOMES

data = load_data()
borough_data = data.copy()

np.random.seed(RANDOM_STATE)

In [ ]:
# Fit borough models and examine residual spatial autocorrelation
walkability_sd, borough_rows, borough_residual_rows = data['walkability_score'].std(ddof = 1), [], []
for adjustment, covariates in ADJUSTMENT_SETS.items():
    for outcome in DISEASES:
        model_data = borough_data.loc[complete_case_mask(borough_data, outcome, ['LAD22NM', *covariates])]

        base = smf.ols(
            f'{outcome} ~ walkability_score + ' + ' + '.join(covariates), data = model_data
        ).fit()
        fixed = smf.ols(
            f'{outcome} ~ walkability_score + ' + ' + '.join(covariates) + ' + C(LAD22NM)', data = model_data
        ).fit()

        base_coef = abs(base.params['walkability_score'])
        fixed_coef = abs(fixed.params['walkability_score'])
        fixed_attenuation_percent = 100 * (1 - fixed_coef / base_coef)

        for label, model, attenuation_percent in [
            ('Adjusted', base, 0),
            ('Borough fixed effects', fixed, fixed_attenuation_percent)
        ]:
            ci = model.conf_int().loc['walkability_score'] * walkability_sd

            borough_rows.append(
                {
                    'adjustment': adjustment,
                    'outcome': outcome,
                    'model': label,
                    'n_lsoas': int(model.nobs),
                    'coefficient_per_sd': model.params['walkability_score'] * walkability_sd,
                    'ci_lower_per_sd': ci.iloc[0],
                    'ci_upper_per_sd': ci.iloc[1],
                    'p_value': model.pvalues['walkability_score'],
                    'attenuation_percent': attenuation_percent
                }
            )

        weights = libpysal.weights.Queen.from_dataframe(model_data, use_index = True)
        weights.transform = 'r'

        residual_moran = esda.Moran(fixed.resid, weights)

        borough_residual_rows.append(
            {
                'adjustment': adjustment,
                'outcome': outcome,
                'residual_moran_i': residual_moran.I,
                'permutation_p_value': residual_moran.p_sim
            }
        )

borough_fe_results = pd.DataFrame(borough_rows)
borough_residual_results = pd.DataFrame(borough_residual_rows)

borough_fe_results.to_csv(OUT.borough_models_csv, index = False)
borough_residual_results.to_csv(OUT.borough_residual_moran_csv, index = False)

display(borough_fe_results.round(4))
display(borough_residual_results.round(4))